# Time-to-platinum: adjust-only vs full models without vitals, means only

Fit **XGBoost survival:cox, elastic-net Cox, and predictive checklists** on the full ADT cohort at **0, 90, and 180 days** on the reduced
predictor set from `10_platinum_feature_ablation.ipynb` (without vitals, `mean-n-obs` statistics), and compare two
uses of the same columns, with paired-patient bootstrap 95% CIs on each difference. Two arms × two models × three
landmarks gives 12 survival fits. The checklists are two further runs of six landmark/horizon cells each, compared in
the [last section](#Predictive-checklists).

| Column group | Full (`--adjust-only-mode none`) | Adjust-only (`--adjust-only-mode drop`) |
|---|---|---|
| Age, lab `__mean` | Predictors | Predictors |
| `__n_observations`, `__missing` | Predictors | Fit in training, removed from the final model |
| Panel version, somatic burden | Elastic-net: predictors. XGBoost: unused | Fit in training, removed from the final model |

The adjust-only arm is therefore the means-only model: means and age are the only terms in its coefficients,
trees, and scores. "Full" names the model, not notebook 10's `all` statistic set.

All arms retain the same inputs, prebuilt patient splits, landmark risk sets, death-censored platinum outcome, and
evaluation horizons. Elastic-net fits adjusters jointly, then zeroes them when scoring. XGBoost fits them as a linear
offset, so its trees never see them. See `03c_multivariate_adjust_only.ipynb` for the method.

**Reading the contrast.** A lower adjust-only AUC or C-index estimates how much of the full model's performance came
from how often patients were measured rather than from their values. The XGBoost contrast also adds panel version and
burden to training, which its full arm never uses, so the two models' deltas are not the same contrast.

Full survival fits are the same folders, commands, and settings as notebook 10, so either notebook resumes the other's
work and this one only runs what is missing. Adjust-only fits go to the parallel `local_runs_adt_adjust_only` tree. Requires
the prediction inputs from `01_preprocessing.ipynb`. To add notebook 10's other combinations, widen `VITAL_SETTINGS`
or `STAT_SETTINGS`.

In [ ]:
import json
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve imports whether launched from the repository root or notebook directory.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "COMPASS/survival_analysis/multivariate_analysis.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from COMPASS.survival_analysis.cox_aggregated import ENDPOINTS, ID_COL, VITAL_MEASUREMENTS
from COMPASS.survival_analysis.predictive_checklists.evaluation import paired_bootstrap
from COMPASS.survival_analysis.predictive_checklists.survival import CensoringDistribution
from survival_common.heldout_bootstrap import (
    assert_reproduces_stored, bootstrap_heldout, load_test_predictions, percentile_ci,
)
from survival_common.helper import resolve_auc_max_time_units

SURVIVAL_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
INPUTS_DIR = SURVIVAL_ROOT / "prediction_inputs_adt"
FULL_ROOT = SURVIVAL_ROOT / "local_runs_adt" / "platinum_feature_ablation"  # shared with notebook 10
ADJUST_ONLY_ROOT = SURVIVAL_ROOT / "local_runs_adt_adjust_only" / "platinum_feature_ablation"
CHECKLIST_ROOT = SURVIVAL_ROOT / "predictive_checklists" / "platinum_adjust_only_vs_full"
CHECKLIST_CONFIG = REPO_ROOT / "COMPASS/survival_analysis/predictive_checklists/default_config.json"
CHECKLIST_HORIZONS = [365, 1825]  # 1 and 5 years after each landmark
LANDMARKS = [0, 90, 180]
MODELS = ["elastic-net", "xgboost"]
VITAL_SETTINGS = ["without_vitals"]
STAT_SETTINGS = ["mean-n-obs"]
ARMS = {"full": ("none", FULL_ROOT), "adjust_only": ("drop", ADJUST_ONLY_ROOT)}
N_FOLDS = 5
SEED = 42
N_BOOTSTRAP = 1000  # patient resamples of each landmark's test set
OVERWRITE = False  # True also refits the full arm in notebook 10's folders, and both checklist arms
DRY_RUN = False  # print commands without fitting or writing outputs
OUT_OF_FOLD_RISKS = False  # must match notebook 10 to reuse its full fits

TASKS = []
for model, landmark, vitals, stats, arm in product(MODELS, LANDMARKS, VITAL_SETTINGS, STAT_SETTINGS, ARMS):
    mode, root = ARMS[arm]
    folder = root / vitals / stats / model / f"landmark_{landmark}"
    prefix = "cox_agg_multivariable" if model == "elastic-net" else "landmark_xgboost"
    cmd = [sys.executable, str(REPO_ROOT / "COMPASS/survival_analysis/multivariate_analysis.py"),
           "--model", model, "--inputs-dir", str(INPUTS_DIR.resolve()), "--output-dir", str(folder.resolve()),
           "--endpoints", "platinum", "--landmark-days", str(landmark), "--feature-set", "labs",
           "--feature-stats", stats, "--n-folds", str(N_FOLDS), "--seed", str(SEED), "--cohort", "all",
           "--save-test-survival", "--overwrite" if OVERWRITE else "--no-overwrite"]
    if vitals == "without_vitals":
        cmd.append("--exclude-vitals")
    if OUT_OF_FOLD_RISKS:
        cmd.append("--out-of-fold-risks")
    if mode != "none":
        cmd.extend(["--adjust-only-mode", mode])
    TASKS.append(dict(model=model, landmark=landmark, vitals=vitals, stats=stats, arm=arm, mode=mode,
                      folder=folder, prefix=prefix,
                      metrics=folder / f"{prefix}_metrics_landmark{landmark}.csv",
                      risks=folder / f"{prefix}_patient_risks_landmark{landmark}.csv", command=cmd))

CHECKLIST_TASKS = []
for vitals, stats, arm in product(VITAL_SETTINGS, STAT_SETTINGS, ARMS):
    mode = ARMS[arm][0]
    folder = CHECKLIST_ROOT / vitals / stats / arm
    cmd = [sys.executable, "-m", "COMPASS.survival_analysis.predictive_checklists",
           "--config", str(CHECKLIST_CONFIG), "--platinum-inputs", str(INPUTS_DIR.resolve()),
           "--output-dir", str(folder.resolve()), "--endpoints", "platinum",
           "--landmarks", *map(str, LANDMARKS), "--horizons", *map(str, CHECKLIST_HORIZONS),
           "--n-folds", str(N_FOLDS), "--feature-stats", stats, "--adjust-only-mode", mode,
           "--no-baselines", "--overwrite" if OVERWRITE else "--resume"]
    if vitals == "without_vitals":
        cmd.append("--exclude-vitals")
    settings = dict(endpoints=["platinum"], landmarks=LANDMARKS, horizons=CHECKLIST_HORIZONS, n_folds=N_FOLDS,
                    baselines=[], adjust_only_mode=mode, exclude_vitals=vitals == "without_vitals",
                    feature_stats=stats)
    CHECKLIST_TASKS.append(dict(vitals=vitals, stats=stats, arm=arm, mode=mode, folder=folder,
                                settings=settings, command=cmd))
print(f"{len(TASKS)} fits and {len(CHECKLIST_TASKS)} checklist runs; vitals: {', '.join(sorted(VITAL_MEASUREMENTS))}")
display(pd.DataFrame(TASKS)[["model", "landmark", "vitals", "stats", "arm", "folder"]])
display(pd.DataFrame(CHECKLIST_TASKS)[["vitals", "stats", "arm", "mode", "folder"]])

## Check inputs

This checks all landmarks before starting any fit. `n_means` counts the stored predictors in the adjust-only arm and
`n_observation_columns` the stored columns it moves into adjustment. `__missing` indicators are derived later, inside
training folds, so they are not counted here. The source prediction inputs are read directly and never rewritten.

In [ ]:
required = [INPUTS_DIR / "build_manifest.json"]
for lm in LANDMARKS:
    required.extend([INPUTS_DIR / f"aggregated_landmark{lm}.csv",
                     INPUTS_DIR / f"pre_treatment_lab_long_landmark{lm}.csv"])
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Build inputs with notebook 01 or update INPUTS_DIR:\n" + "\n".join(missing))
build_manifest = json.loads((INPUTS_DIR / "build_manifest.json").read_text())
audit = []
for lm in LANDMARKS:
    cols = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{lm}.csv", nrows=0).columns
    features = [c for c in cols if "__" in c]
    if not build_manifest["auc_horizons_by_landmark"][str(lm)]["platinum"]:
        raise ValueError(f"No platinum evaluation horizons for landmark {lm}.")
    for vitals, stats in product(VITAL_SETTINGS, STAT_SETTINGS):
        selected = [c for c in features
                    if (vitals == "with_vitals" or c.rsplit("__", 1)[0] not in VITAL_MEASUREMENTS)
                    and (stats == "all" or c.rsplit("__", 1)[1] in {"mean", "n_observations"})]
        n_observation_columns = sum(c.endswith("__n_observations") for c in selected)
        if len(selected) == n_observation_columns:
            raise ValueError(f"No adjust-only predictors for {lm}, {vitals}, {stats}")
        audit.append(dict(landmark=lm, vitals=vitals, stats=stats, n_candidates=len(selected),
                          n_means=sum(c.endswith("__mean") for c in selected),
                          n_observation_columns=n_observation_columns))
display(pd.DataFrame(audit))

## Fit both arms

Completed fits are skipped, including full fits already made by notebook 10. Each fit writes the usual metrics, held-out
risk scores, feature selections, and coefficients or importance files, plus `run_settings.json` and
`training_time.json`. Changing settings in an existing folder raises an error; use `OVERWRITE = True` or a new root.
`OVERWRITE = True` refits the full arm too, replacing notebook 10's timing record for those fits.

In [ ]:
fit_counts = dict(completed=0, skipped=0)
for task in TASKS:
    print(shlex.join(task["command"]), flush=True)
    if DRY_RUN:
        continue
    folder = task["folder"]
    settings = {k: task[k] for k in ("model", "landmark", "vitals", "stats")}
    settings.update(endpoint="platinum", inputs_dir=str(INPUTS_DIR.resolve()), n_folds=N_FOLDS,
                    seed=SEED, out_of_fold_risks=OUT_OF_FOLD_RISKS, build_manifest=build_manifest)
    if task["mode"] != "none":  # full settings must stay equal to notebook 10's
        settings["adjust_only_mode"] = task["mode"]
    settings_path = folder / "run_settings.json"
    if settings_path.exists() and not OVERWRITE:
        previous = json.loads(settings_path.read_text())
        if previous["settings"] != settings:
            raise ValueError(f"Settings changed under {folder}; use OVERWRITE=True or a new root.")
    timing_path = folder / "training_time.json"
    if task["metrics"].exists() and not OVERWRITE:
        if timing_path.exists() and json.loads(timing_path.read_text())["status"] != "completed":
            raise ValueError(f"Previous fit did not complete under {folder}; use OVERWRITE=True.")
        if not settings_path.exists():
            raise ValueError(f"Cannot verify existing fit under {folder}; use OVERWRITE=True.")
        print(f"[skip] {task['metrics']}")
        fit_counts["skipped"] += 1
        continue
    folder.mkdir(parents=True, exist_ok=True)
    settings_path.write_text(json.dumps({"settings": settings, "command": task["command"]}, indent=2) + "\n")
    timing = dict(status="running", started_at_utc=datetime.now(timezone.utc).isoformat(),
                  finished_at_utc=None, training_seconds=None,
                  scope="full_model_subprocess_including_cv_and_evaluation")
    timing_path.write_text(json.dumps(timing, indent=2) + "\n")
    started = time.perf_counter()
    try:
        subprocess.run(task["command"], cwd=REPO_ROOT, check=True)
    except BaseException as exc:
        timing["status"] = "failed"
        timing["error_type"] = type(exc).__name__
        raise
    else:
        timing["status"] = "completed"
    finally:
        timing["training_seconds"] = time.perf_counter() - started
        timing["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        timing_path.write_text(json.dumps(timing, indent=2) + "\n")
    print(f"[completed] Training job: {timing['training_seconds'] / 60:.2f} min", flush=True)
    fit_counts["completed"] += 1
print(f"{fit_counts['completed']} fits completed, {fit_counts['skipped']} skipped")

## Confirm the adjust-only arm is means-only

Reads each finished adjust-only fit and raises an error if an adjuster reached its coefficients or importances, since
the comparison below would then be invalid. `non_mean_predictors` lists anything reported other than age and `__mean`
terms. Expected to be empty for `mean-n-obs`, and to hold the other statistics for `all`.

In [ ]:
ADJUSTER_SUFFIXES = ("__n_observations", "__missing")
ADJUSTER_PREFIXES = ("PANEL_VERSION_", "SOMATIC_N_GENES_ALTERED")

def is_adjuster(name):
    return str(name).endswith(ADJUSTER_SUFFIXES) or str(name).startswith(ADJUSTER_PREFIXES)

def is_mean_or_age(name):
    return str(name).endswith("__mean") or "__" not in str(name)

checks = []
for task in TASKS:
    if task["arm"] != "adjust_only" or not task["metrics"].exists():
        continue
    landmark, folder = task["landmark"], task["folder"]
    if task["model"] == "elastic-net":
        terms = pd.read_csv(folder / f"cox_agg_multivariable_landmark{landmark}.csv")
        used = terms.loc[terms["coef"] != 0, "feature"]
        adjusters = pd.read_csv(folder / f"cox_agg_multivariable_adjusters_landmark{landmark}.csv")
        n_adjusters = len(adjusters)
    else:
        terms = pd.read_csv(folder / f"landmark_xgboost_feature_importance_landmark{landmark}.csv")
        used = terms.loc[terms["gain"] > 0, "feature"]
        n_adjusters = np.nan  # the offset is not a reported table
    leaked = sorted(set(terms.loc[terms["feature"].map(is_adjuster), "feature"]))
    if leaked:
        raise ValueError(f"Adjusters reported under {folder}: {leaked}")
    checks.append(dict(model=task["model"], landmark=landmark, vitals=task["vitals"], stats=task["stats"],
                       n_reported_terms=len(terms), n_used_terms=used.nunique(), n_adjuster_terms=n_adjusters,
                       non_mean_predictors=sorted(set(terms.loc[~terms["feature"].map(is_mean_or_age), "feature"]))))
display(pd.DataFrame(checks))

## Held-out comparison

Compare mean AUC(t), C-index, and integrated Brier on each landmark's test set. Higher AUC/C-index and lower Brier
are better. Deltas are **adjust-only minus full** within model and landmark: a negative AUC/C-index delta or a positive
Brier delta favors the full model. Positive training-time deltas mean adjust-only took longer, and timing depends on
hardware and concurrent workload. Different landmarks have different at-risk populations. Rows with only one finished
arm keep their values and show a missing delta.

In [ ]:
ARM_KEYS = ["model", "landmark", "vitals", "stats"]
rows = []
for task in TASKS:
    row = {k: task[k] for k in (*ARM_KEYS, "arm")}
    row.update(status="missing", source_path=str(task["metrics"]), risks_path=str(task["risks"]))
    if task["metrics"].exists():
        frame = pd.read_csv(task["metrics"])
        frame = frame.loc[frame["endpoint"] == "platinum"]
        if len(frame) != 1:
            raise ValueError(f"Expected one platinum metrics row: {task['metrics']}")
        row.update(frame.iloc[0].to_dict())
        row.update({k: task[k] for k in (*ARM_KEYS, "arm")})
        row["status"] = "ok"
    timing_path = task["folder"] / "training_time.json"
    timing = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    row["timing_status"] = timing.get("status", "not_recorded")
    row["training_seconds"] = timing.get("training_seconds", np.nan) if timing.get("status") == "completed" else np.nan
    row["training_minutes"] = row["training_seconds"] / 60
    if timing and timing["status"] != "completed":
        row["status"] = "training_" + timing["status"]
    rows.append(row)
summary = pd.DataFrame(rows)
metrics = ["test_mean_auc_t", "test_c_index", "test_integrated_brier"]
for col in [*metrics, "n_test", "n_events_test"]:
    if col not in summary:
        summary[col] = np.nan
display(summary[[*ARM_KEYS, "arm", "status", "n_test", "n_events_test", *metrics, "training_minutes", "timing_status"]])
if not DRY_RUN:
    ADJUST_ONLY_ROOT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_summary.csv", index=False)

not_ok = summary.loc[summary["status"] != "ok", [*ARM_KEYS, "arm", "status"]]
if not not_ok.empty:
    print(f"{len(not_ok)} of {len(summary)} fits are not comparable yet:")
    display(not_ok)

ok = summary.loc[summary["status"] == "ok"]
compared = [*metrics, "training_minutes"]
comparison = pd.DataFrame(columns=ARM_KEYS)
if not ok.empty:
    counts = ok.groupby("landmark")[["n_test", "n_events_test"]].nunique()
    if (counts > 1).any().any():
        raise ValueError("Test cohort counts differ across arms; inspect inputs and stale outputs before comparing.")
    by_arm = [ok.loc[ok["arm"] == arm].set_index(ARM_KEYS)[compared].add_suffix(f"_{arm}") for arm in ARMS]
    comparison = by_arm[0].join(by_arm[1], how="outer")
    for metric in compared:
        comparison[f"delta_{metric}"] = comparison[f"{metric}_adjust_only"] - comparison[f"{metric}_full"]
    comparison = comparison.reset_index()
    display(comparison.round(3))

## Bootstrap confidence intervals

Each landmark's test patients are resampled with replacement `N_BOOTSTRAP` times, and both arms are rescored on the
same draws, using the same seeds as notebook 10. Each delta's interval comes from the per-draw difference between the
arms on the same patients, so it cannot be read off the two arms' separate intervals.

- **Same metrics:** recomputed values must match each stored metric, or this section stops.
- **What the intervals cover:** test-sampling uncertainty only. Fitted models and tuned hyperparameters are held fixed.
- **Reading them:** 95% percentile intervals, unadjusted for multiplicity. Six model/landmark groups and three metrics
  give 18 delta intervals, so about one would exclude zero by chance alone.
- **Integrated Brier:** needs the survival probabilities saved by `--save-test-survival`. A full fit from before notebook
  10 added that flag still gets C-index and AUC(t) intervals, and its Brier interval stays blank until it is refit.

In [ ]:
def arm_intervals(draws, keys, metrics):
    """Each fit's percentile interval for each metric."""
    rows = []
    for (*key, arm), group in draws.groupby([*keys, "arm"]):
        row = dict(zip(keys, key), arm=arm)
        for metric in metrics:
            row[f"{metric}_ci_low"], row[f"{metric}_ci_high"], row[f"{metric}_n_valid_draws"] = percentile_ci(group[metric])
        rows.append(row)
    columns = [f"{metric}_{part}" for metric in metrics for part in ("ci_low", "ci_high", "n_valid_draws")]
    return pd.DataFrame(rows, columns=[*keys, "arm", *columns])


def add_delta_intervals(comparison, draws, keys, metrics):
    """Intervals on adjust-only minus full over the same draws; rows with one arm stay blank."""
    if comparison.empty:
        return comparison
    fit_draws = {key: group.set_index("draw") for key, group in draws.groupby([*keys, "arm"])}
    rows = []
    for key in comparison[keys].itertuples(index=False, name=None):
        row = dict(zip(keys, key))
        if (*key, "full") in fit_draws and (*key, "adjust_only") in fit_draws:
            difference = fit_draws[(*key, "adjust_only")][metrics] - fit_draws[(*key, "full")][metrics]
            for metric in metrics:
                row[f"delta_{metric}_ci_low"], row[f"delta_{metric}_ci_high"], _ = percentile_ci(difference[metric])
        rows.append(row)
    columns = [f"delta_{metric}_ci_{bound}" for metric in metrics for bound in ("low", "high")]
    return comparison.merge(pd.DataFrame(rows, columns=[*keys, *columns]), on=keys, validate="one_to_one")


def delta_with_ci(row, metric):
    delta, low, high = row[f"delta_{metric}"], row[f"delta_{metric}_ci_low"], row[f"delta_{metric}_ci_high"]
    return f"{delta:+.3f} ({low:+.3f}, {high:+.3f})" if np.isfinite(low) else f"{delta:+.3f} (no CI)"


endpoint = ENDPOINTS["platinum"]
time_unit_days = int(build_manifest["auc_time_unit_days"])
max_time_unit = resolve_auc_max_time_units(build_manifest, None)

draw_frames = []
for landmark, landmark_fits in ok.groupby("landmark", sort=False):
    columns = {endpoint["duration_col"]: "duration_days", endpoint["event_col"]: "event"}
    inputs = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{landmark}.csv", usecols=["split", *columns])
    reference = inputs.loc[inputs["split"].isin(["train", "valid"])].rename(columns=columns)
    horizons = np.asarray(build_manifest["auc_horizons_by_landmark"][str(landmark)]["platinum"], dtype=float)
    for key, group in landmark_fits.groupby(ARM_KEYS, sort=False):
        fits = {row["arm"]: load_test_predictions(row["risks_path"]) for _, row in group.iterrows()}
        observed, fit_draw_frame = bootstrap_heldout(
            fits, reference=reference, horizons=horizons, time_unit_days=time_unit_days,
            max_time_unit=max_time_unit, n_boot=N_BOOTSTRAP, seed=SEED + int(landmark), id_col=ID_COL,
        )
        stored = group.set_index("arm")
        for _, row in observed.iterrows():
            assert_reproduces_stored(row, stored.loc[row["fit"]], context=f"{key} {row['fit']}")
        draw_frames.append(fit_draw_frame.rename(columns={"fit": "arm"}).assign(**dict(zip(ARM_KEYS, key))))
draws = (pd.concat(draw_frames, ignore_index=True) if draw_frames
         else pd.DataFrame(columns=[*ARM_KEYS, "arm", "draw", *metrics]))

ci_summary = ok[[*ARM_KEYS, "arm", "n_test", "n_events_test", *metrics]].merge(
    arm_intervals(draws, ARM_KEYS, metrics), on=[*ARM_KEYS, "arm"], validate="one_to_one",
)
comparison = add_delta_intervals(comparison, draws, ARM_KEYS, metrics)

if not comparison.empty:
    display(comparison[ARM_KEYS].assign(**{f"delta_{metric}": comparison.apply(delta_with_ci, axis=1, metric=metric)
                                           for metric in metrics}))
    no_brier = ci_summary["test_integrated_brier_ci_low"].isna().sum()
    if no_brier:
        print(f"{no_brier}/{len(ci_summary)} fits have no integrated Brier CI: "
              "refit them with OVERWRITE = True to save test survival.")
    if not DRY_RUN:
        comparison.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_comparison.csv", index=False)
        ci_summary.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_bootstrap_ci.csv", index=False)
        draws.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_bootstrap_draws.csv", index=False)

In [ ]:
POINT_COLOR, GRID, INK = "#2a78d6", "#c3c2b7", "#0b0b0b"


def delta_forest(plotted, panel_col, panels, metric, xlabel, title, filename):
    """Paired delta and CI, one panel per `panels` entry; rows follow plotted["row"] from the top."""
    row_labels = list(dict.fromkeys(plotted["row"]))
    positions = {label: i for i, label in enumerate(row_labels)}
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 1.6 + 0.6 * len(row_labels)),
                             sharex=True, sharey=True, squeeze=False)
    for ax, (panel, label) in zip(axes[0], panels.items()):
        sub = plotted.loc[plotted[panel_col] == panel]
        y = sub["row"].map(positions)
        ax.axvline(0, color=INK, lw=0.8, zorder=1)
        ax.hlines(y, sub[f"delta_{metric}_ci_low"], sub[f"delta_{metric}_ci_high"],
                  color=POINT_COLOR, lw=2, zorder=2)
        ax.scatter(sub[f"delta_{metric}"], y, s=80, color=POINT_COLOR, edgecolor="white", linewidth=1.5, zorder=3)
        ax.set_title(label, loc="left")
        ax.set_xlabel(xlabel)
        ax.grid(axis="x", color=GRID, lw=0.5, alpha=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.tick_params(axis="y", length=0)
    axes[0][0].set_yticks(range(len(row_labels)), row_labels)
    axes[0][0].set_ylim(len(row_labels) - 0.5, -0.5)
    fig.suptitle(title, x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    if not DRY_RUN:
        fig.savefig(ADJUST_ONLY_ROOT / filename, dpi=180, bbox_inches="tight")
    plt.show()


if not comparison.empty:
    multiple_sets = comparison[["vitals", "stats"]].drop_duplicates().shape[0] > 1
    plotted = comparison.sort_values(["landmark", "vitals", "stats"])
    plotted = plotted.assign(row=[f"+{row.landmark}d | {row.vitals} | {row.stats}" if multiple_sets else f"+{row.landmark}d"
                                  for row in plotted.itertuples()])
    delta_forest(plotted, "model", {model: model for model in MODELS}, "test_mean_auc_t",
                 "Mean AUC(t), adjust-only minus full",
                 "Change in held-out mean AUC(t), with paired bootstrap 95% CI", "adjust_only_vs_full_mean_auc.png")

## Predictive checklists

This section fits one M-of-N checklist per landmark and horizon in each arm, using `default_config.json` for platinum
only, at 1 and 5 years after each landmark: six cells per run, on the same vitals and statistic settings as above.

| Candidate items | Full (`--adjust-only-mode none`) | Adjust-only (`--adjust-only-mode drop`) |
|---|---|---|
| Age, lab `__mean` thresholds | Candidates | Candidates |
| `__n_observations` thresholds, `is missing` | Candidates | Never items; adjust calibration only |
| Panel version, somatic burden | Unused | Never items; adjust calibration only |

**Reading the contrast.** The two survival models adjust their fitted effects, but the checklist optimizer cannot take an
offset, so adjusters never enter item selection. The AUC delta therefore measures only what is lost by removing count
and missingness items from the candidate pool. The calibrator sees the adjusters, so the Brier delta also reflects
fitting calibration conditional on them and then predicting at their training mean. When both arms choose the same items
their scores are identical, so that cell's AUC delta and its interval are exactly zero while its Brier delta can still
move. A zero-width row in the forest plot means the same checklist, not a failed comparison.

**Comparators are off** (`--no-baselines`). In drop mode the checklist comparators are restricted but not adjusted, so an
adjust-only vs full row for them would not be the contrast this notebook reports. The survival models above fill that
role with properly adjusted arms.

**Run time.** Each cell solves up to 16 checklist fits (three sizes × five folds, plus the final fit) at up to 60 seconds
each. The worst case is about 16 minutes per cell, or a little over 3 hours for both arms. To shorten a run, reduce `CHECKLIST_HORIZONS`.

Finished runs are skipped after their settings are checked; inputs are not compared, so rerun with `OVERWRITE = True`
after rebuilding them. An interrupted run resumes from its last finished horizon,
and the checklist CLI refuses to resume if its settings, inputs, or source code have changed; use `OVERWRITE = True`,
which archives the previous results, or a new `CHECKLIST_ROOT`.

In [ ]:
checklist_counts = dict(completed=0, skipped=0)
FINISHED_STATUSES = {"complete", "complete_with_unsupported_cells"}
MANIFEST_DEFAULTS = dict(adjust_only_mode="none", exclude_vitals=False, feature_stats="all")  # omitted when unset
for task in CHECKLIST_TASKS:
    print(shlex.join(task["command"]), flush=True)
    if DRY_RUN:
        continue
    manifest_path = task["folder"] / "run_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    if manifest.get("status") in FINISHED_STATUSES and not OVERWRITE:
        config = {**MANIFEST_DEFAULTS, **manifest["config"]}
        if {key: config[key] for key in task["settings"]} != task["settings"]:
            raise ValueError(f"Settings changed under {task['folder']}; use OVERWRITE=True or a new CHECKLIST_ROOT.")
        print(f"[skip] {task['folder']}")
        checklist_counts["skipped"] += 1
        continue
    task["folder"].mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    subprocess.run(task["command"], cwd=REPO_ROOT, check=True)
    print(f"[completed] Checklist run: {(time.perf_counter() - started) / 60:.2f} min", flush=True)
    checklist_counts["completed"] += 1
print(f"{checklist_counts['completed']} checklist runs completed, {checklist_counts['skipped']} skipped")

### Confirm each checklist stayed in its pool

Reads every finished checklist and raises an error if an item is outside its arm's candidate pool, since the comparison
below would then be invalid. The table lists the items each arm chose.

In [ ]:
def out_of_pool_items(items, task):
    """Names of items that the arm's vitals, statistic, and adjust-only settings should have excluded."""
    invalid = []
    for item in items:
        source, _, stat = str(item["feature"]).rpartition("__")
        if not source:  # age
            continue
        adjuster = stat == "n_observations" or item["op"] == "is_missing"
        if ((task["mode"] == "drop" and adjuster)
                or (task["vitals"] == "without_vitals" and source in VITAL_MEASUREMENTS)
                or (task["stats"] == "mean-n-obs" and stat not in {"mean", "n_observations"})):
            invalid.append(item["name"])
    return invalid


CHECKLIST_KEYS = ["vitals", "stats", "landmark", "horizon"]
item_rows = []
for task, landmark, horizon in product(CHECKLIST_TASKS, LANDMARKS, CHECKLIST_HORIZONS):
    path = task["folder"] / "platinum" / f"landmark_{landmark}" / "full" / f"horizon_{horizon}" / "checklist.json"
    if not path.exists():
        continue
    checklist = json.loads(path.read_text())
    invalid = out_of_pool_items(checklist["items"], task)
    if invalid:
        raise ValueError(f"Checklist items outside the candidate pool in {path}: {invalid}")
    item_rows.append(dict(vitals=task["vitals"], stats=task["stats"], landmark=landmark, horizon=horizon,
                          arm=task["arm"], threshold=checklist["threshold"], n_items=len(checklist["items"]),
                          items="; ".join(item["name"] for item in checklist["items"])))
checklist_items = pd.DataFrame(item_rows, columns=[*CHECKLIST_KEYS, "arm", "threshold", "n_items", "items"])
with pd.option_context("display.max_colwidth", None):
    display(checklist_items)

### Held-out checklist comparison

Compare each checklist's AUC and Brier score at its horizon on the test set. Higher AUC and lower Brier are better, and
deltas are **adjust-only minus full**. A cell can be unsupported or fail in one arm, for example when a late horizon has
too few patients; its reason is listed below, and any row with one finished arm keeps its values and shows a missing delta.

Both arms are rescored on the same patient draws, with each landmark using the same seed as the survival bootstrap. The
recomputed values must match each stored metric, or this section stops. Intervals cover test sampling only, with the
checklist and calibrator held fixed, and they are unadjusted 95% percentile intervals. Six cells and two metrics give
12 correlated delta intervals, so one could exclude zero by chance alone.

`n_items` is a count, so it has no interval. Each cell's `solver_status` and `solver_optimality_gap` are in the summary
CSV: a checklist stopped by the time limit is the best one found, not a proven optimum.

In [ ]:
CHECKLIST_COLUMNS = ["status", "reason", "n", "n_cases", "n_controls", "n_unknown", "auc", "brier", "n_items",
                     "threshold", "selected_max_items", "solver_status", "solver_optimality_gap"]
rows = []
for task in CHECKLIST_TASKS:
    path = task["folder"] / "metrics.csv"
    stored = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["model", "landmark_days", "horizon_days"])
    stored = stored.loc[stored["model"] == "checklist"].set_index(["landmark_days", "horizon_days"])
    for landmark, horizon in product(LANDMARKS, CHECKLIST_HORIZONS):
        row = dict(vitals=task["vitals"], stats=task["stats"], landmark=landmark, horizon=horizon, arm=task["arm"],
                   status="missing", folder=str(task["folder"]))
        if (landmark, horizon) in stored.index:
            row.update(stored.loc[(landmark, horizon)].reindex(CHECKLIST_COLUMNS).to_dict())
        rows.append(row)
checklist_summary = pd.DataFrame(rows).reindex(columns=[*CHECKLIST_KEYS, "arm", "folder", *CHECKLIST_COLUMNS])
if not DRY_RUN:
    ADJUST_ONLY_ROOT.mkdir(parents=True, exist_ok=True)
    checklist_summary.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_checklist_summary.csv", index=False)
    checklist_items.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_checklist_items.csv", index=False)

not_ok = checklist_summary.loc[checklist_summary["status"] != "ok", [*CHECKLIST_KEYS, "arm", "status", "reason"]]
if not not_ok.empty:
    print(f"{len(not_ok)} of {len(checklist_summary)} checklist cells are not comparable:")
    display(not_ok)

checklist_ok = checklist_summary.loc[checklist_summary["status"] == "ok"]
checklist_metrics = ["auc", "brier"]
compared = [*checklist_metrics, "n_items"]
checklist_comparison = pd.DataFrame(columns=CHECKLIST_KEYS)
if not checklist_ok.empty:
    by_arm = [checklist_ok.loc[checklist_ok["arm"] == arm].set_index(CHECKLIST_KEYS)[compared].add_suffix(f"_{arm}")
              for arm in ARMS]
    checklist_comparison = by_arm[0].join(by_arm[1], how="outer")
    for metric in compared:
        checklist_comparison[f"delta_{metric}"] = (checklist_comparison[f"{metric}_adjust_only"]
                                                   - checklist_comparison[f"{metric}_full"])
    checklist_comparison = checklist_comparison.reset_index()

draw_frames = []
for landmark, landmark_cells in checklist_ok.groupby("landmark", sort=False):
    outcome_cols = [endpoint["duration_col"], endpoint["event_col"]]
    inputs = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{landmark}.csv", usecols=[ID_COL, "split", *outcome_cols],
                         dtype={ID_COL: "string"})
    development = inputs.loc[inputs["split"].isin(["train", "valid"])]
    test = inputs.loc[inputs["split"] == "test"]
    censoring = CensoringDistribution.fit(*(development[col].to_numpy() for col in outcome_cols))
    for key, cell in landmark_cells.groupby(CHECKLIST_KEYS, sort=False):
        horizon = key[CHECKLIST_KEYS.index("horizon")]
        fits = {}
        for row in cell.itertuples():
            path = Path(row.folder) / "platinum" / f"landmark_{landmark}" / "full" / f"horizon_{horizon}" / "predictions.csv"
            saved = pd.read_csv(path, dtype={ID_COL: "string"})
            saved = saved.loc[saved["dataset"] == "test"].set_index(ID_COL).reindex(test[ID_COL])
            if saved["score"].isna().any():
                raise ValueError(f"Test patients are missing from {path}.")
            fits[row.arm] = (saved["score"].to_numpy(), saved["flag"].to_numpy(), saved["probability"].to_numpy())
        observed, draw_rows = paired_bootstrap(
            *(test[col].to_numpy() for col in outcome_cols), fits, horizon, censoring,
            bootstrap_samples=N_BOOTSTRAP, seed=SEED + int(landmark),
        )
        stored = cell.set_index("arm")
        for arm, values in observed.items():
            expected = stored.loc[arm, checklist_metrics].astype(float).to_numpy()
            if not np.allclose([values[metric] for metric in checklist_metrics], expected):
                raise ValueError(f"Recomputed metrics differ from stored ones for {key} {arm}.")
        draw_frames.append(pd.DataFrame(draw_rows).rename(columns={"fit": "arm"}).assign(**dict(zip(CHECKLIST_KEYS, key))))
checklist_draws = (pd.concat(draw_frames, ignore_index=True) if draw_frames
                   else pd.DataFrame(columns=[*CHECKLIST_KEYS, "arm", "draw", *checklist_metrics]))

checklist_ci = checklist_ok[[*CHECKLIST_KEYS, "arm", *compared]].merge(
    arm_intervals(checklist_draws, CHECKLIST_KEYS, checklist_metrics), on=[*CHECKLIST_KEYS, "arm"], validate="one_to_one",
)
checklist_comparison = add_delta_intervals(checklist_comparison, checklist_draws, CHECKLIST_KEYS, checklist_metrics)
if not checklist_comparison.empty:
    display(checklist_comparison[[*CHECKLIST_KEYS, "auc_full", "auc_adjust_only", "n_items_full", "n_items_adjust_only"]].assign(
        **{f"delta_{metric}": checklist_comparison.apply(delta_with_ci, axis=1, metric=metric)
           for metric in checklist_metrics}).round(3))
    if not DRY_RUN:
        checklist_comparison.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_checklist_comparison.csv", index=False)
        checklist_ci.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_checklist_bootstrap_ci.csv", index=False)
        checklist_draws.to_csv(ADJUST_ONLY_ROOT / "adjust_only_vs_full_checklist_bootstrap_draws.csv", index=False)

In [ ]:
if not checklist_comparison.empty:
    multiple_sets = checklist_comparison[["vitals", "stats"]].drop_duplicates().shape[0] > 1
    plotted = checklist_comparison.sort_values(["horizon", "vitals", "stats"])
    plotted = plotted.assign(row=[f"{row.horizon}d | {row.vitals} | {row.stats}" if multiple_sets else f"{row.horizon}d"
                                  for row in plotted.itertuples()])
    delta_forest(plotted, "landmark", {landmark: f"+{landmark}d landmark" for landmark in LANDMARKS}, "auc",
                 "Checklist AUC, adjust-only minus full",
                 "Change in held-out checklist AUC by horizon, with paired bootstrap 95% CI",
                 "adjust_only_vs_full_checklist_auc.png")